In [1]:
import pandas as pd

df = pd.read_csv("customer_churn_sample (2).csv")

df.head()

,CustomerID,Gender,Age,TenureMonths,SubscriptionType,MonthlyCharges,TotalCharges,ContractType,SupportTickets,PaymentMethod,Churn
0,CUST-1001,Female,34,12,Basic,49.99,599.88,Month-to-Month,3,Credit Card,Yes
1,CUST-1002,Male,45,24,Pro,79.99,1919.76,One Year,1,Bank Transfer,No
2,CUST-1003,Female,29,6,Basic,49.99,299.94,Month-to-Month,5,UPI,Yes
3,CUST-1004,Male,52,36,Enterprise,149.99,5399.64,Two Year,0,Credit Card,No
4,CUST-1005,Female,41,8,Pro,79.99,639.92,Month-to-Month,4,Debit Card,Yes


In [2]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Rows: 15
Columns: 11

Missing values:
CustomerID          0
Gender              0
Age                 0
TenureMonths        0
SubscriptionType    0
MonthlyCharges      0
TotalCharges        0
ContractType        0
SupportTickets      0
PaymentMethod       0
Churn               0
dtype: int64

Duplicate rows: 0


In [3]:
print(df.columns.tolist())

['CustomerID', 'Gender', 'Age', 'TenureMonths', 'SubscriptionType', 'MonthlyCharges', 'TotalCharges', 'ContractType', 'SupportTickets', 'PaymentMethod', 'Churn']


In [4]:
data = df.copy()

data["Churn"] = data["Churn"].str.strip().str.title()
data["ContractType"] = data["ContractType"].str.strip()
data["PaymentMethod"] = data["PaymentMethod"].str.strip()

print(data.head())

  CustomerID  Gender  Age  TenureMonths SubscriptionType  MonthlyCharges  \
0  CUST-1001  Female   34            12            Basic           49.99   
1  CUST-1002    Male   45            24              Pro           79.99   
2  CUST-1003  Female   29             6            Basic           49.99   
3  CUST-1004    Male   52            36       Enterprise          149.99   
4  CUST-1005  Female   41             8              Pro           79.99   

   TotalCharges    ContractType  SupportTickets  PaymentMethod Churn  
0        599.88  Month-to-Month               3    Credit Card   Yes  
1       1919.76        One Year               1  Bank Transfer    No  
2        299.94  Month-to-Month               5            UPI   Yes  
3       5399.64        Two Year               0    Credit Card    No  
4        639.92  Month-to-Month               4     Debit Card   Yes  


In [5]:
total_customers = len(data)
churned_customers = (data["Churn"] == "Yes").sum()
retained_customers = (data["Churn"] == "No").sum()

churn_rate = churned_customers / total_customers * 100
retention_rate = retained_customers / total_customers * 100

revenue_at_risk = data.loc[data["Churn"] == "Yes", "MonthlyCharges"].sum()

average_tenure = data["TenureMonths"].mean()

print("Total Customers:", total_customers)
print("Churn Rate:", round(churn_rate, 2), "%")
print("Retention Rate:", round(retention_rate, 2), "%")
print("Revenue at Risk: ₹", round(revenue_at_risk, 2))
print("Average Tenure:", round(average_tenure, 2), "months")

Total Customers: 15
Churn Rate: 46.67 %
Retention Rate: 53.33 %
Revenue at Risk: ₹ 409.93
Average Tenure: 18.8 months


In [6]:
kpi_summary = pd.DataFrame({
    "KPI": [
        "Churn Rate",
        "Retention Rate",
        "Revenue at Risk",
        "Average Tenure"
    ],
    "Value": [
        f"{churn_rate:.2f}%",
        f"{retention_rate:.2f}%",
        f"₹{revenue_at_risk:.2f}",
        f"{average_tenure:.2f} months"
    ]
})

kpi_summary


,KPI,Value
0,Churn Rate,46.67%
1,Retention Rate,53.33%
2,Revenue at Risk,₹409.93
3,Average Tenure,18.80 months


In [7]:
!pip -q install plotly ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 46.1 MB/s eta 0:00:00


In [8]:
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

In [9]:
from IPython.display import Markdown, display

display(Markdown("# Business KPI Dashboard"))
display(Markdown("### Customer Churn, Retention, Revenue Risk & Customer Segments"))

# Business KPI Dashboard

### Customer Churn, Retention, Revenue Risk & Customer Segments

In [10]:
contract_filter = widgets.Dropdown(
    options=["All"] + sorted(data["ContractType"].unique().tolist()),
    value="All",
    description="Contract:"
)

tenure_filter = widgets.Dropdown(
    options=["All", "0–12 months", "13–24 months", "25+ months"],
    value="All",
    description="Tenure:"
)

payment_filter = widgets.Dropdown(
    options=["All"] + sorted(data["PaymentMethod"].unique().tolist()),
    value="All",
    description="Payment:"
)

display(contract_filter, tenure_filter, payment_filter)

Dropdown(description='Contract:', options=('All', 'Month-to-Month', 'One Year', 'Two Year'), value='All')

Dropdown(description='Tenure:', options=('All', '0–12 months', '13–24 months', '25+ months'), value='All')

Dropdown(description='Payment:', options=('All', 'Bank Transfer', 'Credit Card', 'Debit Card', 'UPI'), value='…

In [11]:
def filter_data():
    filtered = data.copy()

    if contract_filter.value != "All":
        filtered = filtered[
            filtered["ContractType"] == contract_filter.value
        ]

    if payment_filter.value != "All":
        filtered = filtered[
            filtered["PaymentMethod"] == payment_filter.value
        ]

    if tenure_filter.value == "0–12 months":
        filtered = filtered[filtered["TenureMonths"] <= 12]

    elif tenure_filter.value == "13–24 months":
        filtered = filtered[
            (filtered["TenureMonths"] >= 13) &
            (filtered["TenureMonths"] <= 24)
        ]

    elif tenure_filter.value == "25+ months":
        filtered = filtered[filtered["TenureMonths"] >= 25]

    return filtered

In [12]:
output = widgets.Output()

def update_dashboard(change=None):
    with output:
        clear_output(wait=True)

        filtered = filter_data()

        if len(filtered) == 0:
            print("No customers match the selected filters.")
            return

        total = len(filtered)
        churned = (filtered["Churn"] == "Yes").sum()
        retained = (filtered["Churn"] == "No").sum()

        churn = churned / total * 100
        retention = retained / total * 100
        risk = filtered.loc[
            filtered["Churn"] == "Yes",
            "MonthlyCharges"
        ].sum()
        tenure = filtered["TenureMonths"].mean()

        print(f"Customers: {total}")
        print(f"Churn Rate: {churn:.2f}%")
        print(f"Retention Rate: {retention:.2f}%")
        print(f"Revenue at Risk: ₹{risk:.2f}")
        print(f"Average Tenure: {tenure:.2f} months")

        churn_chart = px.pie(
            filtered,
            names="Churn",
            title="Churn vs Retained Customers"
        )

        churn_chart.show()

        contract_chart = px.bar(
            filtered.groupby("ContractType", as_index=False)["MonthlyCharges"].sum(),
            x="ContractType",
            y="MonthlyCharges",
            title="Monthly Revenue by Contract Type",
            labels={
                "ContractType": "Contract Type",
                "MonthlyCharges": "Monthly Charges (₹)"
            }
        )

        contract_chart.show()

        tenure_chart = px.bar(
            filtered.groupby("ContractType", as_index=False)["TenureMonths"].mean(),
            x="ContractType",
            y="TenureMonths",
            title="Average Tenure by Contract Type",
            labels={
                "ContractType": "Contract Type",
                "TenureMonths": "Average Tenure (Months)"
            }
        )

        tenure_chart.show()

        payment_chart = px.bar(
            filtered.groupby("PaymentMethod", as_index=False)["MonthlyCharges"].sum(),
            x="PaymentMethod",
            y="MonthlyCharges",
            title="Monthly Charges by Payment Method",
            labels={
                "PaymentMethod": "Payment Method",
                "MonthlyCharges": "Monthly Charges (₹)"
            }
        )

        payment_chart.show()


contract_filter.observe(update_dashboard, names="value")
tenure_filter.observe(update_dashboard, names="value")
payment_filter.observe(update_dashboard, names="value")

display(output)

update_dashboard()

Output()

In [13]:
segment_data = data.copy()

segment_data["CustomerSegment"] = pd.cut(
    segment_data["TenureMonths"],
    bins=[-1, 12, 24, float("inf")],
    labels=["New", "Established", "Long-term"]
)

segment_summary = (
    segment_data
    .groupby("CustomerSegment", observed=False)
    .agg(
        Customers=("CustomerID", "count"),
        Churned=("Churn", lambda x: (x == "Yes").sum()),
        AvgMonthlyCharge=("MonthlyCharges", "mean")
    )
    .reset_index()
)

segment_summary["ChurnRate"] = (
    segment_summary["Churned"] /
    segment_summary["Customers"] * 100
)

segment_summary

,CustomerSegment,Customers,Churned,AvgMonthlyCharge,ChurnRate
0,New,6,6,54.99,100.0
1,Established,5,1,67.99,20.0
2,Long-term,4,0,149.99,0.0


In [14]:
fig = px.bar(
    segment_summary,
    x="CustomerSegment",
    y="ChurnRate",
    text="ChurnRate",
    title="Churn Rate by Customer Segment",
    labels={
        "CustomerSegment": "Customer Segment",
        "ChurnRate": "Churn Rate (%)"
    }
)

fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.show()

## Key Insights

- Churn rate shows the share of customers who left during the observation period.
- Retention rate shows the share of customers who remained active.
- Revenue at risk represents the monthly charges associated with churned customers.
- Average tenure shows the typical number of months customers have remained subscribed.
- Customer segments help compare churn patterns across different tenure groups.

### Business Interpretation

The dashboard can help a subscription team identify customer groups with higher churn and associated monthly revenue at risk. These observations are descriptive and should not be interpreted as proof that a particular contract, payment method, or tenure group causes churn.

## Data Quality Check

- Dataset contains 15 customer records.
- Dataset contains 11 columns.
- Missing values were checked before analysis.
- Duplicate rows were checked.
- CustomerID is treated as an identifier, not as a measure.
- Churn, contract type, and payment method values were cleaned for consistent formatting.

In [15]:
data.to_csv("customer_churn_cleaned.csv", index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
